# Step 4: Supabase Relational Ingestion

This notebook reads `leetcode_clean.jsonl` and `company_questions.jsonl` and pushes the data to **Supabase (PostgreSQL)**.

*(Note: Vector embeddings are handled in a separate notebook `kaggle_qdrant_ingest.ipynb` to be run on Kaggle for GPU acceleration).*

**Required:** A `.env` file in this folder with `SUPABASE_URL` and `SUPABASE_SERVICE_KEY`.

In [ ]:
# Cell 1 — Install Dependencies & Load Env
!pip install -q supabase python-dotenv pandas

import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Clients
from supabase import create_client, Client

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_KEY")

assert SUPABASE_URL and SUPABASE_KEY, "Missing Supabase credentials in .env"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

PROJECT_ROOT = Path(r"C:/PRAKSHIT/VS CODE/Prep Assistant Project/placed_in")
LC_FILE = PROJECT_ROOT / "data" / "my" / "leetcode_clean.jsonl"
CO_FILE = PROJECT_ROOT / "data" / "my" / "company_questions.jsonl"

print("✅ Supabase Client initialized and paths verified.")

In [ ]:
# Cell 2 — Ingest leetcode_clean.jsonl to Supabase
print("Loading leetcode problems...")
lc_records = []
with LC_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            lc_records.append(json.loads(line))

# Map JSONL fields to Supabase SQL schema
supabase_lc_records = []
for r in lc_records:
    supabase_lc_records.append({
        "id": r["id"],
        "internal_id": r.get("internalId"),
        "title": r.get("title"),
        "slug": r.get("slug"),
        "difficulty": r.get("difficulty"),
        "category": r.get("category"),
        "is_paid_only": r.get("isPaidOnly", False),
        "content": r.get("content"),
        "topic_tags": r.get("topicTags", []),
        "hints": r.get("hints", []),
        "example_testcases": r.get("exampleTestcases"),
        "similar_questions": r.get("similarQuestions", []),
        "likes": r.get("likes"),
        "dislikes": r.get("dislikes"),
        "ac_rate": r.get("acRate"),
        "total_accepted": r.get("totalAccepted"),
        "total_submission": r.get("totalSubmission"),
    })

batch_size = 500
print(f"Inserting {len(supabase_lc_records)} problems into Supabase...")
for i in range(0, len(supabase_lc_records), batch_size):
    batch = supabase_lc_records[i:i+batch_size]
    res = supabase.table("lc_problems").upsert(batch).execute()
    print(f"  Upserted batch {i//batch_size + 1} ({min(i+batch_size, len(supabase_lc_records))}/{len(supabase_lc_records)})")
print("✅ Problems successfully ingested to Supabase.")

In [ ]:
# Cell 3 — Ingest company_questions.jsonl to Supabase
print("Loading company questions...")
co_records = []
with CO_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            co_records.append(json.loads(line))

supabase_co_records = []
lc_ids = {r["id"] for r in supabase_lc_records} # to avoid foreign key constraints

for r in co_records:
    # Only include records whose problem_id actually exists in lc_problems to prevent FK errors
    if r["problemId"] in lc_ids:
        supabase_co_records.append({
            "company": r.get("company"),
            "problem_id": r.get("problemId"),
            "slug": r.get("slug"),
            "title": r.get("title"),
            "difficulty": r.get("difficulty"),
            "acceptance": r.get("acceptance"),
            "frequency": r.get("frequency"),
            "windows": r.get("windows", [])
        })

batch_size = 1000
print(f"Inserting {len(supabase_co_records)} company records into Supabase...")
for i in range(0, len(supabase_co_records), batch_size):
    batch = supabase_co_records[i:i+batch_size]
    # Note: Requires UNIQUE(company, problem_id) constraint in Supabase
    res = supabase.table("lc_company_questions").upsert(batch, on_conflict="company,problem_id").execute()
    if i % 10000 == 0:
        print(f"  Upserted {min(i+batch_size, len(supabase_co_records))}/{len(supabase_co_records)}")
print("✅ Company questions successfully ingested to Supabase.")